# Notebook 02 — Build the 120-Image Manifest

**Objective.** Build and inspect the candidate 120-image benchmark manifest (six buckets).

**Inputs.** GQA scene graphs.

**Expected outputs.** Candidate manifest CSV, bucket distribution, review checklist.

_Companion notebook: runnable top-to-bottom, relative paths, no hidden state, saves artifacts to `results/`._

In [ ]:
import sys, os
# Reusable logic lives in src/grounded_memory/. Notebooks call it.
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
from pathlib import Path
RESULTS = Path('..') / 'results'
from grounded_memory import __version__
print('grounded_memory', __version__)

## Build candidate manifest

Score images per bucket and greedily fill six 20-image buckets.

In [ ]:
from pathlib import Path

from grounded_memory.config import load_config
from grounded_memory.data.gqa import scene_graph_path, load_scene_graphs
from grounded_memory.data.selection import build_candidate_manifest, BUCKETS

# gqa.root is repo-root-relative (e.g. "./data/raw/gqa") -- resolve it against
# the repo root explicitly, since the notebook kernel's cwd is notebooks/, not
# the repo root (same fix applied in the manual-review setup cell below).
cfg = load_config('datasets')
gqa_root = Path(cfg['gqa']['root'])
if not gqa_root.is_absolute():
    gqa_root = (Path('..').resolve() / gqa_root).resolve()
cfg['gqa']['root'] = str(gqa_root)

try:
    sgs = load_scene_graphs(scene_graph_path(cfg, split='val'))
    rows = build_candidate_manifest(sgs, split='val', per_bucket=20)
    from collections import Counter
    print(Counter(r.primary_bucket for r in rows))
except FileNotFoundError as e:
    print('Need GQA first:', e)

## Manual review (REQUIRED before freezing)

This audits the annotations behind each bucket assignment, not just previews
pictures: for each candidate row it shows the raw image (what the VLM sees),
the image with GQA boxes + labels, and the scene-graph triples, side by side.
Uses `grounded_memory.data.review`. Progress saves incrementally to
`gqa_benchmark_v1.review.csv` — the frozen candidate CSV is never touched
until every row has `manually_reviewed=True`.

In [ ]:
from pathlib import Path

from grounded_memory.config import load_config
from grounded_memory.data.gqa import load_scene_graphs, scene_graph_path
from grounded_memory.data.review import draw_boxes, load_manifest, save_manifest, scene_graph_to_triples
from grounded_memory.data.selection import BUCKETS

REPO_ROOT = Path('..').resolve()

cfg = load_config('datasets')
# gqa.root and manifest.path are repo-root-relative (e.g. "./data/raw/gqa") --
# resolve them against REPO_ROOT explicitly, since the notebook kernel's cwd is
# notebooks/, not the repo root.
gqa_root = Path(cfg['gqa']['root'])
if not gqa_root.is_absolute():
    gqa_root = (REPO_ROOT / gqa_root).resolve()
cfg['gqa']['root'] = str(gqa_root)

GQA_IMAGES_DIR = gqa_root / cfg['gqa']['images_dir']
CANDIDATE_CSV = REPO_ROOT / cfg['manifest']['path']
REVIEW_CSV = CANDIDATE_CSV.with_name(CANDIDATE_CSV.stem + '.review' + CANDIDATE_CSV.suffix)
REJECT_LOG_CSV = CANDIDATE_CSV.parent / 'gqa_review_log.csv'
BUCKET_TARGETS = {b: cfg['manifest']['buckets'].get(b, {}).get('count', 20) for b in BUCKETS}

print(f'Candidate manifest: {CANDIDATE_CSV} (exists: {CANDIDATE_CSV.exists()})')
print(f'Review progress:    {REVIEW_CSV} (exists: {REVIEW_CSV.exists()})')
print(f'Reject log:         {REJECT_LOG_CSV} (exists: {REJECT_LOG_CSV.exists()})')
print(f'Images dir:         {GQA_IMAGES_DIR} (exists: {GQA_IMAGES_DIR.exists()})')

In [ ]:
# Resume from review.csv if a prior session already made progress; otherwise
# start from the candidate CSV. The candidate file itself is never written to
# here -- only load_manifest reads it.
if REVIEW_CSV.exists():
    review_rows = load_manifest(REVIEW_CSV)
    print(f'Resumed {len(review_rows)} rows from {REVIEW_CSV.name}')
elif CANDIDATE_CSV.exists():
    review_rows = load_manifest(CANDIDATE_CSV)
    print(f'Loaded {len(review_rows)} candidate rows from {CANDIDATE_CSV.name} (no review progress yet)')
else:
    review_rows = []
    print(f'No manifest found at {CANDIDATE_CSV} -- build the candidate manifest first (see above).')

sg_path = scene_graph_path(cfg, split=cfg['gqa'].get('benchmark_split', 'val'))
try:
    scene_graphs = load_scene_graphs(sg_path)
    print(f'Loaded {len(scene_graphs)} scene graphs from {sg_path}')
except FileNotFoundError as e:
    scene_graphs = {}
    print(f'Need GQA scene graphs first: {e}')

In [ ]:
def render_row(idx: int) -> None:
    """Show the 3-panel review for review_rows[idx]: raw image, boxed image, triples."""
    import matplotlib.pyplot as plt

    row = review_rows[idx]
    image_path = GQA_IMAGES_DIR / f'{row.image_id}.jpg'
    sg = scene_graphs.get(row.scene_graph_id)

    print(f'--- row {idx} / {len(review_rows) - 1}  image_id={row.image_id} ---')
    print(f'primary_bucket={row.primary_bucket}  ({BUCKETS.get(row.primary_bucket, "?")})')
    print(f'objects={row.num_objects}  relations={row.num_relations}  attributes={row.num_attributes}')
    print(f'selection_reason: {row.selection_reason}')
    print(f'manually_reviewed={row.manually_reviewed}  ambiguity_notes={row.ambiguity_notes!r}')

    fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
    if image_path.exists():
        from PIL import Image
        axes[0].imshow(Image.open(image_path))
        axes[0].set_axis_off()
        axes[0].set_title('raw (what the VLM sees)')
        if sg is not None:
            draw_boxes(image_path, sg, ax=axes[1])
            axes[1].set_title('boxes + labels')
        else:
            axes[1].set_axis_off()
            axes[1].text(0.5, 0.5, 'no scene graph loaded', ha='center', va='center')
    else:
        for ax in axes:
            ax.set_axis_off()
        axes[0].text(0.5, 0.5, f'image not found:\n{image_path}', ha='center', va='center', wrap=True)
    plt.show()

    print('\nscene-graph triples:')
    if sg is not None:
        for line in scene_graph_to_triples(sg):
            print(f'  {line}')
    else:
        print('  (none -- scene graph not loaded)')


def _bucket_counts_text() -> str:
    """Live per-bucket counts vs target -- refreshed after every save/reassign/reject."""
    from collections import Counter

    counts = Counter(r.primary_bucket for r in review_rows)
    return '  |  '.join(f'{b}: {counts.get(b, 0)}/{target}' for b, target in BUCKET_TARGETS.items())

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    from grounded_memory.data.review import reject_and_replace

    _idx = 0

    bucket_dropdown = widgets.Dropdown(options=list(BUCKETS), description='bucket:')
    notes_box = widgets.Textarea(description='notes:', layout=widgets.Layout(width='500px'))
    reviewed_checkbox = widgets.Checkbox(description='manually_reviewed')
    prev_btn = widgets.Button(description='< prev')
    next_btn = widgets.Button(description='next >')
    save_btn = widgets.Button(description='Save', button_style='success')
    reject_reason_box = widgets.Text(description='reject reason:', layout=widgets.Layout(width='400px'))
    reject_btn = widgets.Button(description='Reject & replace', button_style='danger')
    position_label = widgets.Label()
    bucket_counts_label = widgets.Label()
    out = widgets.Output()

    def _load_into_widgets(idx):
        row = review_rows[idx]
        bucket_dropdown.value = row.primary_bucket
        notes_box.value = row.ambiguity_notes
        reviewed_checkbox.value = row.manually_reviewed
        position_label.value = f'{idx + 1} / {len(review_rows)}'
        bucket_counts_label.value = _bucket_counts_text()
        out.clear_output(wait=True)
        with out:
            render_row(idx)

    def _on_prev(_):
        global _idx
        _idx = (_idx - 1) % len(review_rows)
        _load_into_widgets(_idx)

    def _on_next(_):
        global _idx
        _idx = (_idx + 1) % len(review_rows)
        _load_into_widgets(_idx)

    def _on_save(_):
        # Plain reassignment (image OK, wrong bucket) goes through here too --
        # editing the bucket dropdown then Save updates the live counts below.
        row = review_rows[_idx]
        row.primary_bucket = bucket_dropdown.value
        row.ambiguity_notes = notes_box.value
        row.manually_reviewed = reviewed_checkbox.value
        save_manifest(review_rows, REVIEW_CSV)
        bucket_counts_label.value = _bucket_counts_text()
        n_reviewed = sum(r.manually_reviewed for r in review_rows)
        with out:
            print(f'Saved row {_idx} -> {REVIEW_CSV.name}  ({n_reviewed}/{len(review_rows)} reviewed so far)')

    def _on_reject(_):
        global _idx
        warning = reject_and_replace(review_rows, _idx, scene_graphs, reject_reason_box.value, REJECT_LOG_CSV)
        save_manifest(review_rows, REVIEW_CSV)
        bucket_counts_label.value = _bucket_counts_text()
        reject_reason_box.value = ''
        if _idx >= len(review_rows):
            _idx = max(0, len(review_rows) - 1)
        with out:
            if warning:
                print(f'WARNING: {warning}')
            elif review_rows:
                print(f'Rejected -> replaced with {review_rows[_idx].image_id}. Review the replacement:')
        if review_rows:
            _load_into_widgets(_idx)

    prev_btn.on_click(_on_prev)
    next_btn.on_click(_on_next)
    save_btn.on_click(_on_save)
    reject_btn.on_click(_on_reject)

    if review_rows:
        _load_into_widgets(_idx)

    display(
        widgets.HBox([prev_btn, next_btn, position_label]),
        bucket_dropdown, notes_box, reviewed_checkbox, save_btn,
        widgets.HBox([reject_reason_box, reject_btn]),
        bucket_counts_label, out,
    )
except ImportError:
    print("ipywidgets not installed -- use the non-widget fallback below "
          "(review_row_fallback / reject_and_replace_fallback) instead of this cell. "
          "Install with the 'notebooks' extra: pip install -e '.[notebooks]'")

In [ ]:
def review_row_fallback(
    idx: int,
    bucket: str | None = None,
    ambiguity_notes: str | None = None,
    manually_reviewed: bool | None = None,
) -> None:
    """Plain-function review path for when ipywidgets isn't installed.

    Call with just `idx` to render the row; pass any of the other arguments to
    update that row's review state and save immediately. Example:
        review_row_fallback(0)                                 # just look
        review_row_fallback(0, manually_reviewed=True)          # mark reviewed + save
        review_row_fallback(0, bucket='C_spatial', ambiguity_notes='occluded')
    """
    row = review_rows[idx]
    changed = bucket is not None or ambiguity_notes is not None or manually_reviewed is not None
    if bucket is not None:
        row.primary_bucket = bucket
    if ambiguity_notes is not None:
        row.ambiguity_notes = ambiguity_notes
    if manually_reviewed is not None:
        row.manually_reviewed = manually_reviewed
    if changed:
        save_manifest(review_rows, REVIEW_CSV)
        print(f'Saved row {idx} -> {REVIEW_CSV.name}')
    render_row(idx)
    print(f'\nbucket counts: {_bucket_counts_text()}')


def reject_and_replace_fallback(idx: int, reason: str) -> None:
    """Plain-function reject-and-replace path for when ipywidgets isn't installed.

    Logs the rejection, swaps in the next-best same-bucket candidate (or removes
    the row and warns if that bucket's pool is exhausted), saves progress, and
    renders whatever ends up at `idx` next. Example:
        reject_and_replace_fallback(0, 'boxes are wrong, image is actually empty')
    """
    from grounded_memory.data.review import reject_and_replace

    warning = reject_and_replace(review_rows, idx, scene_graphs, reason, REJECT_LOG_CSV)
    save_manifest(review_rows, REVIEW_CSV)
    if warning:
        print(f'WARNING: {warning}')
    elif idx < len(review_rows):
        render_row(idx)
    print(f'\nbucket counts: {_bucket_counts_text()}')

## Freeze the manifest

Only writes the frozen `gqa_benchmark_v1.csv` once every row's
`manually_reviewed` is `True`. Otherwise it just reports how many rows are
still outstanding — review progress stays in `gqa_benchmark_v1.review.csv`.

In [ ]:
from collections import Counter

from grounded_memory.data.review import load_rejection_log

print('Reminder: GQA scene graphs are a reference annotation, not omniscient '
      'truth -- a missing annotation is not the same as something being false.')

n_total = len(review_rows)
n_reviewed = sum(r.manually_reviewed for r in review_rows)
bucket_counts = Counter(r.primary_bucket for r in review_rows)
bucket_mismatches = {
    b: (bucket_counts.get(b, 0), target)
    for b, target in BUCKET_TARGETS.items()
    if bucket_counts.get(b, 0) != target
}
n_rejected = len(load_rejection_log(REJECT_LOG_CSV))

print('\nPer-bucket counts (actual / target):')
for b, target in BUCKET_TARGETS.items():
    flag = '' if bucket_counts.get(b, 0) == target else '  <-- MISMATCH'
    print(f'  {b}: {bucket_counts.get(b, 0)} / {target}{flag}')
print(f'Total rejections logged: {n_rejected}')

if n_total == 0:
    print('\nNo rows loaded -- nothing to freeze.')
elif n_reviewed < n_total:
    print(f'\n{n_total - n_reviewed} / {n_total} rows still need manually_reviewed=True. '
          f'Not freezing. Progress is saved in {REVIEW_CSV.name}.')
elif bucket_mismatches:
    print('\nBucket counts do not match targets -- not freezing. Reject-and-replace or '
          'reassign rows until every bucket is exactly at its target count.')
else:
    save_manifest(review_rows, CANDIDATE_CSV)
    print(f'\nAll {n_total} rows reviewed and every bucket is at its target count. '
          f'Froze {CANDIDATE_CSV.name}.')

## Findings

_Fill in 3–5 concrete findings after running this notebook._

1. …
2. …
3. …

> Do not populate tables with invented numbers. If the full model/data is unavailable, leave clearly-labeled placeholders.